In [1]:
import pandas as pd
from collections import defaultdict

High-level MP from db HMD_HumanPhenotype.rpt

In [2]:
mgi_mp_db1 = pd.read_csv("db_1_hum_mouse_genes.csv", delimiter="\t")

In [3]:
mgi_mp_db1.head()

,Unnamed: 0,gene_human,entrez_id_human,gene_mouse,MGI,MP
0,0,A1BG,1,A1bg,MGI:2152878,NaN
1,1,A1CF,29974,A1cf,MGI:1917115,"MP:0005367, MP:0005369, MP:0005370, MP:0005376..."
2,2,A2M,2,A2m,MGI:2449119,NaN
3,3,A3GALT2,127550,A3galt2,MGI:2685279,NaN
4,4,A4GALT,53947,A4galt,MGI:3512453,"MP:0005376, MP:0005386, MP:0010768"


In [4]:
mgi_mp_db1.shape[0] # MGI number

29686

In [5]:
mgi_mp_db1.MGI.nunique() # unique MGIs

20131

In [6]:
mgi_mp_db1[mgi_mp_db1.MP.astype(str) != 'nan'].shape[0] # MGI with high-level MP number | MGI-MP mappings

13634

In [7]:
# list of all non-unique MP terms

mp_list_db_1 = []
for i in list(mgi_mp_db1.MP):
    mp_list_db_1.extend(str(i).strip("'").split(", "))

In [8]:
len(mp_list_db_1) # number of all non-unique MPs

94794

In [9]:
len(set(mp_list_db_1)) # number of unique MPs

28

Low-level MP from MGI_PhenoGenoMP.rpt

In [10]:
mgi_mp_pheno_geno = pd.read_csv("../databases/MGI_PhenoGenoMP.rpt", delimiter="\t", header=None)\
    .rename(columns={3: "MP", 5: "MGI"})

In [11]:
mgi_mp_pheno_geno

,0,1,2,MP,4,MGI
0,Rb1<tm1Tyj>/Rb1<tm1Tyj>,Rb1<tm1Tyj>,involves: 129S2/SvPas,MP:0000600,12529408,MGI:97874
1,Rb1<tm1Tyj>/Rb1<tm1Tyj>,Rb1<tm1Tyj>,involves: 129S2/SvPas,MP:0001716,16449662,MGI:97874
2,Rb1<tm1Tyj>/Rb1<tm1Tyj>,Rb1<tm1Tyj>,involves: 129S2/SvPas,MP:0001698,16449662,MGI:97874
3,Rb1<tm1Tyj>/Rb1<tm1Tyj>,Rb1<tm1Tyj>,involves: 129S2/SvPas,MP:0001092,16449662,MGI:97874
4,Rb1<tm1Tyj>/Rb1<tm1Tyj>,Rb1<tm1Tyj>,involves: 129S2/SvPas,MP:0000961,16449662,MGI:97874
...,...,...,...,...,...,...
367165,"E2f1<Tg(Wnt1-cre)2Sor>/E2f1<+>,Snrpb<em1Lajm>/...",Snrpb<+>|E2f1<Tg(Wnt1-cre)2Sor>|E2f1<+>|Trp53<...,involves: 129P2/OlaHsd * C3H * C57BL/6 * CD1,MP:0000458,35593225,MGI:101941|MGI:98342|MGI:98834
367166,"E2f1<Tg(Wnt1-cre)2Sor>/E2f1<+>,Snrpb<em1Lajm>/...",Snrpb<+>|E2f1<Tg(Wnt1-cre)2Sor>|E2f1<+>|Trp53<...,involves: 129P2/OlaHsd * C3H * C57BL/6 * CD1,MP:0000445,35593225,MGI:101941|MGI:98342|MGI:98834
367167,"E2f1<Tg(Wnt1-cre)2Sor>/E2f1<+>,Snrpb<em1Lajm>/...",Snrpb<+>|E2f1<Tg(Wnt1-cre)2Sor>|E2f1<+>|Trp53<...,involves: 129P2/OlaHsd * C3H * C57BL/6 * CD1,MP:0002639,35593225,MGI:101941|MGI:98342|MGI:98834
367168,"E2f1<Tg(Wnt1-cre)2Sor>/E2f1<+>,Snrpb<em1Lajm>/...",Snrpb<+>|E2f1<Tg(Wnt1-cre)2Sor>|E2f1<+>|Trp53<...,involves: 129P2/OlaHsd * C3H * C57BL/6 * CD1,MP:0008271,35593225,MGI:101941|MGI:98342|MGI:98834


In [12]:
mgi_mp_pheno_geno.MP.nunique()

10870

In [13]:
mgi_mp_pheno_geno_dict = defaultdict(set)
f = open("../databases/MGI_PhenoGenoMP.rpt", "r")
for line in f:
    line_data = line.rstrip().split("\t")
    mgis = (line_data[5].split("|"))
    mp = line_data[3]
    for mgi in mgis:
        mgi_mp_pheno_geno_dict[mgi].add(mp)
f.close()

In [15]:
len(mgi_mp_pheno_geno_dict) # number of mgi-mps

22791

In [25]:
f = open("mgi_mp_pheno_geno.tsv", "w")
for mgi in mgi_mp_pheno_geno_dict:
    mps = ", ".join(list(mgi_mp_pheno_geno_dict[mgi]))
    f.write(f"{mgi}\t{mps}\n")
f.close()

In [28]:
mgi_mp_pheno_geno_db = pd.read_csv("mgi_mp_pheno_geno.tsv", delimiter="\t", header=None)\
    .rename(columns={0: "MGI", 1: "MP"})

Merging databases

In [37]:
db_4_corr_without_human = pd.merge(mgi_mp_db1, mgi_mp_pheno_geno_db, how='left', on = 'MGI')\
    .rename(columns={"MP_x": "MP_high_level", "MP_y": "MP_low_level"})

In [43]:
db_4_corr_without_human[(db_4_corr_without_human.MP_high_level.astype(str) == 'nan') & (db_4_corr_without_human.MP_low_level.astype(str) != 'nan')]

,Unnamed: 0,gene_human,entrez_id_human,gene_mouse,MGI,MP_high_level,MP_low_level
110,110,ABLIM1,3983,Ablim1,MGI:1194500,NaN,MP:0002169
114,114,ABR,29,Abr,MGI:107771,NaN,"MP:0002978, MP:0001394, MP:0001399, MP:0006089..."
124,124,ACAA1,30,Acaa1b,MGI:3605455,NaN,MP:0002169
182,182,ACOT6,641372,Acot6,MGI:1921287,NaN,MP:0002169
213,213,ACSM3,6296,Acsm3,MGI:99538,NaN,MP:0002169
...,...,...,...,...,...,...,...
29464,29464,ZNF860,344787,Zfp998,MGI:1924053,NaN,"MP:0012167, MP:0020845"
29532,29532,ZNF878,729747,Zfp998,MGI:1924053,NaN,"MP:0012167, MP:0020845"
29606,29606,ZNF93,81931,Zfp998,MGI:1924053,NaN,"MP:0012167, MP:0020845"
29639,29639,ZRSR2,8233,Zrsr2-ps1,MGI:98885,NaN,"MP:0002169, MP:0011094"


In [45]:
db_4_corr_without_human.to_csv("db_4_corr_without_human.tsv", sep='\t')